# Практична робота №1
## Розгортання середовища та знайомство з IoT-даними

У цій роботі ви:

- перевірите роботу контейнеризованого середовища;
- ознайомитеся зі структурою навчального проєкту;
- прочитаєте файли JSON, CSV і JSON Lines;
- визначите склад IoT-системи;
- виконаєте базові підрахунки без очищення даних;
- збережете результати у визначеному форматі.

> Виконуйте комірки послідовно зверху вниз. Не змінюйте файли в каталозі `data/input`.

## Дані студента

**Прізвище, ім’я та по батькові:** _вкажіть тут_  
**Група:** _вкажіть тут_  
**Дата виконання:** _вкажіть тут_  

Ідентифікатор варіанта та студентський ідентифікатор далі будуть прочитані з `metadata.json`.

## Межі цієї роботи

У практичній роботі №1 **не потрібно**:

- очищувати або виправляти події;
- шукати дублікати;
- перевіряти допустимі межі значень;
- перевіряти правильність часових міток;
- виявляти невідомі пристрої чи непідтримувані метрики;
- змінювати початкові файли.

Ці завдання виконуватимуться в наступній практичній роботі з якості даних.

## 1. Перевірка робочого середовища

In [ ]:
import json
import sys
from pathlib import Path

import duckdb
import polars as pl
import pyarrow as pa

print("Python:", sys.version.split()[0])
print("Polars:", pl.__version__)
print("DuckDB:", duckdb.__version__)
print("PyArrow:", pa.__version__)

In [ ]:
PROJECT_DIR = Path("/workspace")
INPUT_DIR = PROJECT_DIR / "data" / "input"
WORKING_DIR = PROJECT_DIR / "data" / "working"
RESULTS_DIR = PROJECT_DIR / "results" / "practical_01"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_DIR:", PROJECT_DIR)
print("INPUT_DIR:", INPUT_DIR)
print("WORKING_DIR:", WORKING_DIR)
print("RESULTS_DIR:", RESULTS_DIR)

In [ ]:
# Повна автоматична перевірка середовища.
!python /workspace/scripts/check_environment.py

### Результат перевірки

Запишіть, чи завершилася перевірка повідомленням **«Середовище готове до роботи»**.

**Відповідь:** _вкажіть тут_

## 2. Структура навчального проєкту

### Завдання 2.1

Програмно виведіть вміст кореневого каталогу `/workspace`. Для кожного об’єкта покажіть його назву та тип: файл або каталог.

In [ ]:
# TODO: виведіть об'єкти, що розташовані безпосередньо в PROJECT_DIR.
# Підказка: використайте PROJECT_DIR.iterdir(), path.name та path.is_dir().


### Завдання 2.2

Коротко поясніть призначення каталогів:

- `data/input`;
- `data/working`;
- `notebooks`;
- `results`;
- `src`.

**Відповідь:**

_впишіть пояснення тут_

## 3. Перевірка комплектності індивідуального пакета

Перш ніж аналізувати дані, необхідно переконатися, що всі файли варіанта наявні та не порожні.


In [ ]:
REQUIRED_FILES = [
    "raw_iot_events.jsonl",
    "device_registry.csv",
    "metric_catalog.csv",
    "device_type_metrics.csv",
    "metadata.json",
]

# Приклад роботи з одним шляхом.
example_path = INPUT_DIR / "metadata.json"

print("Шлях:", example_path)
print("Файл існує:", example_path.exists())
print(
    "Розмір у байтах:",
    example_path.stat().st_size if example_path.exists() else None,
)


### Завдання 3.1

Для кожного файла зі списку `REQUIRED_FILES` визначте:

- назву файла;
- чи існує він;
- розмір у байтах;
- чи є він порожнім.

Сформуйте Polars DataFrame `file_inventory` зі стовпцями:

| filename | exists | size_bytes | is_empty |
|---|---:|---:|---:|

Використайте цикл, `Path.exists()` і `Path.stat().st_size`.


In [ ]:
# TODO: виконайте завдання відповідно до умови вище.


In [ ]:
# Самоперевірка завдання 3.1.
EXPECTED_INVENTORY_COLUMNS = [
    "filename",
    "exists",
    "size_bytes",
    "is_empty",
]

assert isinstance(file_inventory, pl.DataFrame), (
    "file_inventory повинен бути Polars DataFrame"
)
assert file_inventory.columns == EXPECTED_INVENTORY_COLUMNS, (
    f"Очікувані стовпці: {EXPECTED_INVENTORY_COLUMNS}"
)
assert file_inventory.height == len(REQUIRED_FILES), (
    "У таблиці має бути один рядок для кожного обов'язкового файла"
)
assert file_inventory["exists"].all(), (
    "Не всі обов'язкові файли знайдено"
)
assert (file_inventory["size_bytes"] > 0).all(), (
    "Один або кілька вхідних файлів порожні"
)
assert not file_inventory["is_empty"].any(), (
    "Один або кілька вхідних файлів позначено як порожні"
)

print("[ OK ] Усі обов'язкові файли знайдено і вони не порожні.")


### Висновок до розділу

Вкажіть:

1. Чи всі необхідні файли наявні?
2. Який файл має найбільший розмір?
3. Чому файл із подіями значно більший за довідники?

**Відповідь:**

_впишіть відповідь тут_


## 4. Метадані індивідуального варіанта

Файл `metadata.json` описує не окремі події, а весь виданий студенту набір даних.


### Завдання 4.1

Прочитайте `metadata.json` за допомогою стандартного модуля `json`.

Результат збережіть у змінній `metadata`. Вона повинна містити словник Python.


In [ ]:
# TODO: відкрийте файл у режимі читання з кодуванням UTF-8


In [ ]:
# Самоперевірка завдання 4.1.
EXPECTED_METADATA_FIELDS = {
    "course",
    "dataset_version",
    "variant_id",
    "group",
    "student_id",
    "student_name",
    "period_start",
    "period_end",
    "timezone",
}

assert isinstance(metadata, dict), (
    "metadata повинен бути словником Python"
)

missing_fields = EXPECTED_METADATA_FIELDS - set(metadata)
assert not missing_fields, (
    f"У metadata відсутні поля: {sorted(missing_fields)}"
)

print("[ OK ] metadata.json успішно прочитано.")
print("Кількість полів:", len(metadata))


### Завдання 4.2

Створіть словник `variant_summary`, вибравши з `metadata` такі поля:

- `course`;
- `dataset_version`;
- `variant_id`;
- `group`;
- `student_id`;
- `student_name`;
- `period_start`;
- `period_end`;
- `timezone`.

Не вписуйте значення вручну — отримайте їх зі словника `metadata`.


In [ ]:
# TODO: виконайте завдання відповідно до умови вище.


In [ ]:
# Самоперевірка завдання 4.2.
assert set(variant_summary) == EXPECTED_METADATA_FIELDS, (
    "variant_summary повинен містити всі дев'ять потрібних полів"
)
assert variant_summary["variant_id"] == metadata["variant_id"]
assert variant_summary["student_id"] == metadata["student_id"]

print("[ OK ] Короткий опис варіанта сформовано.")
print(
    f"Варіант: {variant_summary['variant_id']} | "
    f"Студент: {variant_summary['student_name']}"
)


### Висновок до розділу

Коротко поясніть:

1. Чим `metadata.json` відрізняється від файла подій?
2. Який часовий період заявлено для вашого варіанта?
3. Для чого в метаданих явно вказана часова зона?

**Відповідь:**

_впишіть відповідь тут_


## 5. Реєстр IoT-пристроїв

Файл `device_registry.csv` описує зареєстровані в системі пристрої. Один рядок відповідає одному пристрою.

### Завдання 5.1

Завантажте файл у Polars DataFrame `device_registry`. Виведіть:

- перші 5 рядків;
- розмір таблиці;
- назви та типи стовпців.

In [ ]:
# TODO: виконайте завдання відповідно до умови вище.


### Завдання 5.2

Визначте:

- кількість зареєстрованих пристроїв;
- кількість типів пристроїв;
- кількість локацій;
- перелік рівнів критичності;
- перелік очікуваних інтервалів передавання даних.

In [ ]:
# TODO: виконайте завдання відповідно до умови вище.


### Завдання 5.3

Побудуйте дві таблиці:

1. `devices_by_type` зі стовпцями `device_type`, `device_count`;
2. `devices_by_location` зі стовпцями `location`, `device_count`.

Відсортуйте обидві таблиці за спаданням кількості пристроїв.

In [ ]:
# TODO: виконайте завдання відповідно до умови вище.


### Висновок до розділу

Опишіть склад системи: які типи пристроїв і локації в ній представлені? Що можуть означати поля `criticality` та `expected_interval_sec`?

**Відповідь:** _вкажіть тут_

## 6. Каталог метрик і зв’язки з типами пристроїв

### Завдання 6.1

Завантажте:

- `metric_catalog.csv` у `metric_catalog`;
- `device_type_metrics.csv` у `device_type_metrics`.

Для кожної таблиці виведіть перші рядки, розмір і схему.

In [ ]:
# TODO: виконайте завдання відповідно до умови вище.


### Завдання 6.2

Визначте:

- кількість метрик у каталозі;
- перелік одиниць вимірювання;
- перелік очікуваних типів значень;
- кількість дозволених пар `device_type–metric`.

In [ ]:
# TODO: виконайте завдання відповідно до умови вище.


### Завдання 6.3

Створіть таблицю `metrics_by_device_type` зі стовпцями:

| device_type | metric_count |
|---|---:|

Вона повинна показувати, скільки основних телеметричних метрик підтримує кожен тип пристрою.

In [ ]:
# TODO: виконайте завдання відповідно до умови вище.


### Завдання 6.4

Для кожного типу пристрою виведіть перелік підтримуваних ним метрик.

In [ ]:
# TODO: виконайте завдання відповідно до умови вище.


### Висновок до розділу

Поясніть зв’язок між `device_registry.csv`, `metric_catalog.csv` і `device_type_metrics.csv`.

**Відповідь:** _вкажіть тут_

## 7. Формат JSON Lines і структура події

Файл `raw_iot_events.jsonl` містить необроблені події. У форматі JSON Lines кожен рядок є окремим JSON-об’єктом.

### Завдання 7.1

Прочитайте перші 5 рядків як звичайний текст. Не завантажуйте поки весь файл.

In [ ]:
# TODO: виконайте завдання відповідно до умови вище.


### Завдання 7.2

Перетворіть перший рядок із JSON-тексту на словник Python та виведіть:

- сам словник;
- перелік полів;
- Python-тип значення кожного поля.

In [ ]:
# TODO: виконайте завдання відповідно до умови вище.


### Завдання 7.3

Поясніть семантику полів:

- `event_id`;
- `event_ts`;
- `device_id`;
- `event_type`;
- `metric`;
- `value`.

Також поясніть, чим JSON Lines відрізняється від одного звичайного JSON-масиву.

**Відповідь:** _вкажіть тут_

## 8. Завантаження набору подій

### Завдання 8.1

Завантажте весь файл `raw_iot_events.jsonl` у Polars DataFrame `events` за допомогою `pl.read_ndjson()`.

Після завантаження виведіть:

- перші 5 рядків;
- кількість рядків і стовпців;
- назви та типи стовпців.

> На цьому етапі не виправляйте жодних значень і не видаляйте рядки.

In [ ]:
# TODO: виконайте завдання відповідно до умови вище.


### Завдання 8.2

Визначте лише базові характеристики:

- загальну кількість рядків;
- кількість різних значень `device_id`;
- кількість різних метрик;
- перелік типів подій;
- кількість подій кожного типу.

Не досліджуйте причини можливих розбіжностей із довідниками.

In [ ]:
# TODO: виконайте завдання відповідно до умови вище.


### Висновок до розділу

Які типи подій представлені в наборі? Чим, на вашу думку, відрізняються `telemetry`, `status` і `network`?

**Відповідь:** _вкажіть тут_

## 9. Узагальнена модель IoT-системи

На основі всіх п’яти файлів опишіть шлях даних у системі:

```text
device_type → device → event → metric → value
```

У поясненні вкажіть:

1. де зберігається інформація про конкретний пристрій;
2. де описані метрики та їх одиниці вимірювання;
3. де задано, які телеметричні метрики підтримує тип пристрою;
4. де зберігаються фактичні події;
5. як за `device_id` подію можна пов’язати з типом і локацією пристрою.

**Відповідь:** _вкажіть тут_

## 10. Збереження результатів

У каталозі `results/practical_01` потрібно створити:

```text
system_overview.json
devices_by_type.csv
devices_by_location.csv
metrics_by_device_type.csv
events_by_type.csv
```

Файл `system_overview.json` повинен мати таку структуру:

```json
{
  "dataset_version": "...",
  "variant_id": 0,
  "student_id": "...",
  "registered_devices": 0,
  "device_types": 0,
  "locations": 0,
  "catalog_metrics": 0,
  "allowed_device_type_metric_pairs": 0,
  "raw_event_rows": 0,
  "observed_device_ids": 0,
  "observed_metrics": 0,
  "event_types": []
}
```

Усі значення мають бути отримані програмно з вхідних файлів.

In [ ]:
# TODO: виконайте завдання відповідно до умови вище.


### Самоперевірка вихідних файлів

In [ ]:
EXPECTED_RESULT_FILES = [
    "system_overview.json",
    "devices_by_type.csv",
    "devices_by_location.csv",
    "metrics_by_device_type.csv",
    "events_by_type.csv",
]

EXPECTED_OVERVIEW_KEYS = {
    "dataset_version",
    "variant_id",
    "student_id",
    "registered_devices",
    "device_types",
    "locations",
    "catalog_metrics",
    "allowed_device_type_metric_pairs",
    "raw_event_rows",
    "observed_device_ids",
    "observed_metrics",
    "event_types",
}

missing_result_files = [
    name for name in EXPECTED_RESULT_FILES
    if not (RESULTS_DIR / name).is_file()
]

if missing_result_files:
    print("Відсутні файли:", missing_result_files)
else:
    print("Усі очікувані файли створено.")

    saved_overview = json.loads(
        (RESULTS_DIR / "system_overview.json").read_text(encoding="utf-8")
    )
    missing_keys = sorted(EXPECTED_OVERVIEW_KEYS - set(saved_overview))

    if missing_keys:
        print("У system_overview.json відсутні поля:", missing_keys)
    else:
        print("Структура system_overview.json правильна.")

## 11. Підсумкові висновки

Сформулюйте висновок обсягом приблизно 8–12 речень. Обов’язково зазначте:

- чи вдалося розгорнути й перевірити середовище;
- з яких файлів складається індивідуальний пакет;
- які сутності описують ці файли;
- які типи пристроїв, метрик і подій представлені у вашому варіанті;
- чому вхідні дані відокремлені від робочих і вихідних файлів;
- що нового ви навчилися робити в Python, Polars і JupyterLab.

**Висновок:**

_впишіть текст тут_

## 12. Контрольний список перед здачею

- [ ] Заповнено дані студента.
- [ ] Усі комірки виконано послідовно.
- [ ] У ноутбуці збережено результати виконання комірок.
- [ ] Усі текстові відповіді та висновки заповнено.
- [ ] Початкові файли в `data/input` не змінено.
- [ ] У `results/practical_01` створено п’ять необхідних файлів.
- [ ] Комірка самоперевірки не повідомляє про відсутні файли або поля.